In [0]:
# ==============================================================================
#  DATA MARTS DE INSIGHTS DE NEGÓCIO: ECOMMERCE_CLIENTES
# ==============================================================================
TABELA_ALVO = "ecommerce_enderecos"
print(f"\nGerando Insights de Negócio na memória para {TABELA_ALVO}...")

df_insight = None

# 1. Lê os logs de qualidade DIRETAMENTE do Data Lake para torná-lo independente
if delta_existe(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS):
    df_logs = ler_delta(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)
else:
    df_logs = None

# 2. Processa os insights
if df_logs is not None and df_logs.count() > 0:
    df_insight = df_logs \
        .withColumn("data_execucao", F.to_date("timestamp_execucao")) \
        .groupBy("data_execucao", "regra") \
        .agg(F.sum("qtd_registros_falhos").alias("volume_falhas")) \
        .orderBy(F.col("data_execucao").desc(), F.col("volume_falhas").desc())
else:
    print(f"-> Base limpa! Nenhum erro encontrado para {TABELA_ALVO}.")

# 3. Exibição na Tela
if df_insight is not None:
    print(f"-> Insight gerado com sucesso para {TABELA_ALVO}:")
    display(df_insight)
else:
    print("-> Nenhum insight a ser exibido nesta execução.")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# 3. Exibição na Tela e Gráfico
if df_insight is not None:
    print(f"-> Insight gerado com sucesso para {TABELA_ALVO}:")
    
    # 3.1. Mostra a tabela padrão (se quiser conferir os números exatos)
    display(df_insight)
    
    # 3.2. Converte para Pandas (Seguro, pois é apenas uma tabela agregada pequena)
    df_pd = df_insight.toPandas()
    
    # 3.3. Configura e gera o gráfico
    plt.figure(figsize=(10, 6))
    
    # Exemplo genérico que funciona bem para quase todas as suas tabelas:
    # Eixo X = A primeira coluna do groupBy (Data, Categoria ou Regra)
    # Eixo Y = A coluna que tem a métrica calculada
    coluna_x = df_pd.columns[1] # Pega a coluna de regra/categoria/status
    coluna_y = df_pd.columns[-1] # Pega a última coluna que normalmente é a de volume/faturamento
    
    sns.barplot(data=df_pd, x=coluna_x, y=coluna_y, palette="viridis")
    
    plt.title(f"Análise de Qualidade/Negócio - {TABELA_ALVO}", fontsize=14, fontweight='bold')
    plt.xlabel(coluna_x.replace("_", " ").title(), fontsize=12)
    plt.ylabel(coluna_y.replace("_", " ").title(), fontsize=12)
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("-> Nenhum insight a ser exibido nesta execução.")

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Diagnóstico de reconciliação — ecommerce_enderecos
# MAGIC
# MAGIC Objetivo: descobrir exatamente onde está o resíduo de registros que não
# MAGIC aparece nem na Silver válida, nem na quarentena, nem nos logs de falha.

# COMMAND ----------

# MAGIC %run ../utils/utils

# COMMAND ----------

import pyspark.sql.functions as F

TABELA_ALVO = "ecommerce_enderecos"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Checagem de duplicatas físicas na Bronze

# COMMAND ----------

df_bronze = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)

total_bronze = df_bronze.count()
distintos_bronze = df_bronze.select("id_endereco").distinct().count()
duplicatas_fisicas = total_bronze - distintos_bronze

print(f"Total de linhas físicas na Bronze:      {total_bronze}")
print(f"IDs distintos (id_endereco) na Bronze:   {distintos_bronze}")
print(f"Duplicatas físicas na Bronze:            {duplicatas_fisicas}")

if duplicatas_fisicas > 0:
    print("\nExemplos de id_endereco duplicados na Bronze:")
    display(
        df_bronze.groupBy("id_endereco")
        .count()
        .filter(F.col("count") > 1)
        .orderBy(F.col("count").desc())
        .limit(20)
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Reconciliação Silver + Quarentena vs IDs distintos da Bronze

# COMMAND ----------

qtd_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS).count() \
    if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS) else 0

qtd_quarentena = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS).count() \
    if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS) else 0

soma_processados = qtd_silver + qtd_quarentena
residual_vs_distintos = distintos_bronze - soma_processados
residual_vs_total = total_bronze - soma_processados

print(f"Registros válidos na Silver:                 {qtd_silver}")
print(f"Registros na Quarentena:                     {qtd_quarentena}")
print(f"Soma (Silver + Quarentena):                  {soma_processados}")
print(f"IDs distintos na Bronze:                      {distintos_bronze}")
print(f"Total bruto na Bronze:                        {total_bronze}")
print(f"\nResidual vs IDs DISTINTOS da Bronze:          {residual_vs_distintos}")
print(f"Residual vs TOTAL bruto da Bronze:            {residual_vs_total}")

print("""
Interpretação:
- Se 'Residual vs IDs DISTINTOS' ~= 0  -> o gap era só duplicata física na Bronze
  (nada foi perdido de fato, só contado a mais na Bronze).
- Se 'Residual vs IDs DISTINTOS' > 0   -> ainda existe dado processável que
  não está nem na Silver nem na Quarentena. Ver seção 3 e 4 abaixo.
""")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Quais IDs distintos da Bronze não estão em Silver nem Quarentena?

# COMMAND ----------

df_processados_ids = None

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_s = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS).select("id_endereco")
    df_processados_ids = df_s
else:
    schema_vazio = df_bronze.select("id_endereco").limit(0)
    df_processados_ids = schema_vazio

if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
    df_q = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS).select("id_endereco")
    df_processados_ids = df_processados_ids.union(df_q)

df_processados_ids = df_processados_ids.distinct()

df_bronze_distintos = df_bronze.select("id_endereco").distinct()

df_nao_processados = df_bronze_distintos.join(df_processados_ids, "id_endereco", "left_anti")
qtd_nao_processados = df_nao_processados.count()

print(f"IDs distintos na Bronze que NÃO estão em Silver nem Quarentena: {qtd_nao_processados}")

if qtd_nao_processados > 0:
    print("\nRegistros completos desses IDs (para inspeção manual):")
    display(
        df_bronze.join(df_nao_processados, "id_endereco", "inner")
        .orderBy("bronze_ingested_at")
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Histórico de execuções vs contagem esperada nos logs
# MAGIC
# MAGIC Soma, por execução (run_id), de qtd_registros_total dos logs, comparado
# MAGIC contra o total de itens processados até então — ajuda a ver se alguma
# MAGIC execução específica processou menos do que deveria (indício de gravar_delta
# MAGIC retornando False silenciosamente).

# COMMAND ----------

if delta_existe(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS):
    df_logs = ler_delta(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)

    print("Total processado por execução (run_id), segundo os logs:")
    display(
        df_logs.select("run_id", "qtd_registros_total", "timestamp_execucao")
        .dropDuplicates(["run_id"])
        .orderBy(F.col("timestamp_execucao").desc())
    )

    soma_total_logs = df_logs.select("run_id", "qtd_registros_total") \
        .dropDuplicates(["run_id"]) \
        .agg(F.sum("qtd_registros_total")).collect()[0][0]

    print(f"\nSoma de qtd_registros_total (todas execuções, deduplicado por run_id): {soma_total_logs}")
    print("Compare esse número com 'IDs distintos na Bronze' acima — se baterem,")
    print("as execuções sempre viram todos os registros esperados no momento em que rodaram.")
else:
    print("Tabela dq_monitoring_logs não encontrada.")
